# Download and Load a Kaggle Dataset

Download a dataset with your Kaggle token, unzip it, and read the CSV into Julia with `CSV.jl` + `Tables.jl` (no DataFrames). Set your credentials first via **Set credentials** on the webview, or export `KAGGLE_USERNAME` / `KAGGLE_KEY` yourself.

In [ ]:
# import Pkg; Pkg.add(["HTTP", "JSON3", "ZipFile", "CSV", "Tables"])
using HTTP, JSON3, Base64, ZipFile, CSV, Tables

auth = ["Authorization" => "Basic " * base64encode(ENV["KAGGLE_USERNAME"] * ":" * ENV["KAGGLE_KEY"])]

## List the Files

Check what a dataset contains before downloading it.

In [ ]:
ref = "uciml/iris"   # ownerSlug/datasetSlug
files = JSON3.read(HTTP.get("https://www.kaggle.com/api/v1/datasets/list-files/$ref", auth).body)
for f in files.datasetFiles
    println(f.name, "  ", f.totalBytes, " bytes")
end

## Download the Archive

The download endpoint returns a `.zip` of the dataset's files.

In [ ]:
r = HTTP.get("https://www.kaggle.com/api/v1/datasets/download/$ref", auth)
write("dataset.zip", r.body)
println("saved ", length(r.body), " bytes")

## Read a CSV via Tables.jl

`CSV.File` is a `Tables.jl` source, so it works with the whole Julia data ecosystem without DataFrames.jl.

In [ ]:
z = ZipFile.Reader("dataset.zip")
entry = first(filter(f -> endswith(f.name, ".csv"), z.files))

rows = CSV.File(read(entry))
println("columns: ", Tables.columnnames(rows))
println("rows:    ", length(rows))
println(first(rows))
close(z)

## Summary
- Listing files and downloading need a Kaggle token (`KAGGLE_USERNAME` / `KAGGLE_KEY`).
- The download is a `.zip`; `ZipFile.jl` reads entries without extracting to disk.
- `CSV.File(read(entry))` is a `Tables.jl` source — no DataFrames needed.